In [1]:
# @title 1. Mount Google Drive and Set Up File Paths
from google.colab import drive
import os
from pathlib import Path

# Mount your Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted.


In [11]:
# @title 2. File Paths Set Up

# --- Configuration (UPDATE THESE PATHS) ---
# The path on your Google Drive where your original noisy data is located.
EMPIAR_ID = '10017' # @param {type: "string"}

DATA_DIR_PATH = '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data'     # @param {type: "string"}
DATA_DIR_PATH = DATA_DIR_PATH + f'/EMPIAR-{EMPIAR_ID}/micrographs'
DATA_DIR = Path(DATA_DIR_PATH)

OUTPUT_DIR_PATH ='/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data'  # @param {type: "string"}
OUTPUT_DIR_PATH = OUTPUT_DIR_PATH + f'/EMPIAR-{EMPIAR_ID}/TpzD_micrographs'
OUTPUT_DIR = Path(OUTPUT_DIR_PATH)

TEMP_DIR_PATH = '/content/temp_denoise'                           # @param {type: "string"}
TEMP_DIR = Path(TEMP_DIR_PATH)

SOFTWARE_DIR_PATH = '/content/topaz'                           # @param {type: "string"}
SOFTWARE_DIR = Path(TEMP_DIR_PATH)

!mkdir {OUTPUT_DIR_PATH} -p
!mkdir {TEMP_DIR_PATH} -p
!mkdir {SOFTWARE_DIR_PATH} -p

# Name for the trained model file.
MODEL_NAME = 'denoise_model'                                  # @param {type: "string"}

print(f"\nOriginal Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Temporary Directory: {TEMP_DIR}")

# Create output and temporary directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)


Original Data Directory: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/micrographs
Output Directory: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/TpzD_micrographs
Temporary Directory: /content/temp_denoise


In [3]:
# @title 3. Import Packages
import subprocess
import shutil
import time
from dataclasses import dataclass, field
import sys
import numpy as np

### 4. 🟪 Topaz Installation

In [4]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR_PATH}
  !git clone https://github.com/tbepler/topaz.git
  %cd topaz
  !pip install -e .
  %cd ..

topaz_path = os.path.join(os.getcwd(), 'topaz')
if topaz_path not in sys.path:
    sys.path.insert(0, topaz_path)

print(f"Added {topaz_path} to the Python path.")

/content/topaz
Cloning into 'topaz'...
remote: Enumerating objects: 3509, done.
remote: Counting objects: 100% (573/573), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 3509 (delta 540), reused 490 (delta 490), pack-reused 2936 (from 2)
Receiving objects: 100% (3509/3509), 253.11 MiB | 30.46 MiB/s, done.
Resolving deltas: 100% (2292/2292), done.
/content/topaz/topaz
Obtaining file:///content/topaz/topaz
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for topaz-em (pyproject.toml) ... done
  Created wheel for topaz-em: filename=topaz_em-0.3.16-0.editable-py3-none-any.whl size=23633 sha256=9371225c90d52f7d760187372175542bcb484859b66982d5a5ab3bb99e0ebed6
  Stored in directory: /tmp/pip-ephem-wheel-cache-teo37pq4/wheels/24/41/9d/c6be72d5df4ef1c5f9bc1f70b3f7859ecd3f1b6b79e810aa92
Successfully

In [5]:
# @title Import Topaz Packages

import topaz.mrc as mrc
from topaz.utils.data.loader import load_image

---

In [8]:
# @title 5. Parameter Set Up

# Denoise training parameters
NUM_EPOCHS = 50       # @param {type: "integer"}
ARCH = 'unet'         # @param {type: "string"}
METHOD = 'noise2noise'# @param {type: "string"}

In [9]:
# @title 6. Function Set Up
micrograph_bool = True # @param {type:"boolean"}
train_model = True # @param {type:"boolean"}
use_default_mode = False # @param {type:"boolean"}

@dataclass
class TopazDenoiseConfig:
    """Configuration for the Topaz denoising workflow."""
    software_dir: Path = SOFTWARE_DIR
    data_dir: Path = DATA_DIR
    temp_dir: Path = TEMP_DIR
    output_dir: Path = OUTPUT_DIR
    model_name: str = MODEL_NAME
    model_path: Path = Path(OUTPUT_DIR / (MODEL_NAME + ".pkl") )
    arch: str = ARCH
    num_epochs: int = NUM_EPOCHS


def split_dataset(length : int = None , cfg : TopazDenoiseConfig = TopazDenoiseConfig):
    """Separates files from input_dir into odd and even directories."""
    print("Splitting dataset into odd and even groups...")

    odd_dir = cfg.temp_dir / 'odd'
    even_dir = cfg.temp_dir / 'even'

    # Create temp directories using a single, efficient command
    !mkdir -p {odd_dir} {even_dir}

    # Get the list of all .mrc files
    mrc_files = list(cfg.data_dir.rglob('*.mrc'))
    print(f"Found {len(mrc_files)} .mrc files.")

    if not mrc_files:
        raise FileNotFoundError(
            f"No .mrc files found in {cfg.data_dir} or its subdirectories."
        )
    if length == None:
        length = len(mrc_files)
    ## for each movie frame stack
    ## split it into even/odd frames, sum them, and store these micrographs
    for path in mrc_files[:length]:
        name = os.path.basename(path) # just the name of this micrograph

        ## load the movie frames
        with open(path, 'rb') as f:
            content = f.read()
        frames,_,_ = mrc.parse(content)

        ## split and sum
        if micrograph_bool == False:
            print('Loaded movie frames:', name)
            odd_mic = frames[::2].sum(0)
            even_mic = frames[1:][::2].sum(0)
        else :
            print('Loaded micrographs:', name)
            mic = frames
            top_half = mic[:, :mic.shape[1]//2]
            bottom_half = mic[:, mic.shape[1]//2:]
            odd_mic = top_half
            even_mic = bottom_half

        ## save the split micrographs
        path = odd_dir / name
        with open(path, 'wb') as f:
            mrc.write(f, odd_mic[np.newaxis])


        path = even_dir / name
        with open(path, 'wb') as f:
            mrc.write(f, even_mic[np.newaxis])

    print("Done splitting data.")



def train_model(cfg : TopazDenoiseConfig = TopazDenoiseConfig):
    """Runs the denoise command to train a new model."""
    print("\nTraining a new model...")
    model_prefix = cfg.model_name
    %cd {SOFTWARE_DIR_PATH}
    # Run the topaz command directly with !
    !topaz denoise \
        --method noise2noise \
        -a {cfg.temp_dir}/odd \
        -b {cfg.temp_dir}/even \
        -o {cfg.output_dir} \
        --arch {cfg.arch} \
        --num-epochs {cfg.num_epochs} \
        --save-prefix {model_prefix} \
        --device 0

    return cfg.output_dir / f"{model_prefix}.pkl"


def denoise_original_data(cfg : TopazDenoiseConfig = TopazDenoiseConfig):
    """Applies the trained model to the original dataset."""
    print("\nDenoising the original dataset with the trained model...")

    # Get the list of files to denoise
    files_to_denoise = " ".join([str(f) for f in cfg.data_dir.glob('*') if f.is_file()])
    %cd {cfg.software_dir}
    # Run the denoising command

    if use_default_mode == False:
        !topaz denoise \
            -m {cfg.model_path} \
            -o {cfg.output_dir} \
            --save-prefix "" \
            --device 0 \
            {files_to_denoise}

    else :
        !topaz denoise \
            -o {cfg.output_dir} \
            --save-prefix "" \
            --device 0 \
            {files_to_denoise}

def topaz_denoise_main(length : int = None):
    """Automates the full Noise2Noise denoising workflow."""

    try:
        if train_model == True:
            split_dataset(length)
            train_model()
        denoise_original_data()
        print("\nAll tasks completed successfully!")

    except Exception as e:
        print(f"\nAn error occurred: {e}")

    finally:
        print("Cleanup complete.")

In [ ]:
# @title 7. Operate
data_length_for_training = 10 # @param {type: "integer"}
topaz_denoise_main(data_length_for_training)

---

### Result visualization

In [13]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

name = 'Falcon_2012_06_12-14_33_35_0' # @param {type: "string"}

# load the raw micrograph
mic_raw = np.array(load_image(DATA_DIR_PATH + "/" + name + '.mrc')[0], copy=False)
# load the denoised micrograph
mic_dn = np.array(load_image(OUTPUT_DIR_PATH + "/" + name + '.mrc')[0], copy=False)

# scale them for visualization
mu = mic_dn.mean()
std = mic_dn.std()

mic_raw = (mic_raw - mu)/std
mic_dn = (mic_dn - mu)/std

_,ax = plt.subplots(1,2,figsize=(24,12))

ax[0].imshow(mic_raw, vmin=-4, vmax=4, cmap='Greys_r')
ax[1].imshow(mic_dn, vmin=-4, vmax=4, cmap='Greys_r')

Output hidden; open in https://colab.research.google.com to view.